In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MQA(nn.Module):
    def __init__(self, hidden_dim, num_head, dropout_rate=0.1):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_head = num_head

        assert hidden_dim%num_head==0,"hidden——dim必须整除num——head"

        self.head_dim = hidden_dim//num_head

        self.q_proj = nn.Linear(hidden_dim , hidden_dim)
        self.k_proj = nn.Linear(hidden_dim , self.head_dim)
        self.v_proj = nn.Linear(hidden_dim , self.head_dim)

        self.dropout = nn.Dropout(dropout_rate)
        self.o_proj = nn.Linear(self.hidden_dim,self.hidden_dim)

    def forward(self,x,mask=None):
        # x(b,s,h)
        b,s,_ = x.size()

        Q = self.q_proj(x)
        # (b,s,head_dim)
        K = self.k_proj(x)
        V = self.v_proj(x)
        
        # (b,num_head,s,head_dim)
        Q = Q.view(b,s,self.num_head,-1).transpose(1,2)
        # (b,num_head,s,head_dim)
        K = K.unsqueeze(1).expand(-1,self.num_head,-1,-1)
        V = V.unsqueeze(1).expand(-1,self.num_head,-1,-1)

        # (b,num_head,s,s)
        atten = (Q @ K.transpose(-1,-2)) /(self.head_dim ** 0.5)

        if mask is not None:
            atten = atten.masked_fill(mask==0,float("-inf"))
        
        atten = self.dropout(torch.softmax(atten,dim=-1))
        output = atten @ V

        output = output.transpose(1,2).contiguous().view(b,s,-1)

        output = self.o_proj(output)

        return output



batch_size = 2
seq_len = 10
hidden_size = 256
num_heads = 8

b, s, h = 2, 4, 8  # batch, seq_len, hidden_dim
x = torch.rand(b, s, h)

mask = torch.ones(b,s)
mask[:,3:] = 0

mask = mask.unsqueeze(1).unsqueeze(1)  # 变成 (b, 1, 1, s) -> broadcast 到 num_head, seq_len

mha = MQA(hidden_dim=h, num_head=2)
out = mha(x, mask)
print("输出形状:", out.shape)  # (b, s, h)

输出形状: torch.Size([2, 4, 8])
